In [1]:
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/01 11:07:27 WARN Utils: Your hostname, DESKTOP-JCM8NP2, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/09/01 11:07:27 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/01 11:07:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
df_yellow = spark.read \
    .parquet('data/raw/yellow/2025/11/')

In [5]:
df_yellow.createOrReplaceTempView("yellow")

In [25]:
spark.sql("""
    SELECT
        *, trip_minutes / 60.0 AS trip_hours
    FROM (
        SELECT
            tpep_pickup_datetime,
            tpep_dropoff_datetime,
            total_amount,
            trip_distance,
            fare_amount,
            timestampdiff(
                MINUTE, tpep_pickup_datetime, tpep_dropoff_datetime
            ) AS trip_minutes
        FROM yellow
    )
    ORDER BY trip_hours DESC
    LIMIT 10

""").show()

[Stage 12:===================>                                      (2 + 4) / 6]

+--------------------+---------------------+------------+-------------+-----------+------------+----------+
|tpep_pickup_datetime|tpep_dropoff_datetime|total_amount|trip_distance|fare_amount|trip_minutes|trip_hours|
+--------------------+---------------------+------------+-------------+-----------+------------+----------+
| 2025-11-26 20:22:12|  2025-11-30 15:01:00|       889.1|       121.17|      887.1|        5438| 90.633333|
| 2025-11-27 04:22:41|  2025-11-30 09:19:35|       13.65|         1.08|        7.9|        4616| 76.933333|
| 2025-11-03 10:42:55|  2025-11-06 14:55:45|         4.5|          0.0|        3.0|        4572| 76.200000|
| 2025-11-07 11:23:22|  2025-11-10 08:40:41|       34.25|         6.54|       31.0|        4157| 69.283333|
| 2025-11-18 17:12:47|  2025-11-21 12:17:37|       16.55|         0.76|        9.3|        4024| 67.066667|
| 2025-11-22 17:45:30|  2025-11-25 09:07:36|        14.3|         1.41|       12.8|        3802| 63.366667|
| 2025-11-01 07:44:57|  2025

In [29]:
df_yellow.explain(True)

== Parsed Logical Plan ==
UnresolvedDataSource format: parquet, isStreaming: false, paths: 1 provided

== Analyzed Logical Plan ==
VendorID: int, tpep_pickup_datetime: timestamp_ntz, tpep_dropoff_datetime: timestamp_ntz, passenger_count: bigint, trip_distance: double, RatecodeID: bigint, store_and_fwd_flag: string, PULocationID: int, DOLocationID: int, payment_type: bigint, fare_amount: double, extra: double, mta_tax: double, tip_amount: double, tolls_amount: double, improvement_surcharge: double, total_amount: double, congestion_surcharge: double, Airport_fee: double, cbd_congestion_fee: double
Relation [VendorID#0,tpep_pickup_datetime#1,tpep_dropoff_datetime#2,passenger_count#3L,trip_distance#4,RatecodeID#5L,store_and_fwd_flag#6,PULocationID#7,DOLocationID#8,payment_type#9L,fare_amount#10,extra#11,mta_tax#12,tip_amount#13,tolls_amount#14,improvement_surcharge#15,total_amount#16,congestion_surcharge#17,Airport_fee#18,cbd_congestion_fee#19] parquet

== Optimized Logical Plan ==
Relatio

In [26]:
spark.sql("""
    SELECT
        tpep_pickup_datetime,
        tpep_dropoff_datetime,
        total_amount,
        trip_distance,
        fare_amount,
        timestampdiff(MINUTE, tpep_pickup_datetime, tpep_dropoff_datetime) AS trip_minutes,
        trip_minutes / 60 AS trip_hours
    FROM yellow
    ORDER BY trip_hours DESC
    LIMIT 10

""").show()

[Stage 13:===================>                                      (2 + 4) / 6]

+--------------------+---------------------+------------+-------------+-----------+------------+-----------------+
|tpep_pickup_datetime|tpep_dropoff_datetime|total_amount|trip_distance|fare_amount|trip_minutes|       trip_hours|
+--------------------+---------------------+------------+-------------+-----------+------------+-----------------+
| 2025-11-26 20:22:12|  2025-11-30 15:01:00|       889.1|       121.17|      887.1|        5438|90.63333333333334|
| 2025-11-27 04:22:41|  2025-11-30 09:19:35|       13.65|         1.08|        7.9|        4616|76.93333333333334|
| 2025-11-03 10:42:55|  2025-11-06 14:55:45|         4.5|          0.0|        3.0|        4572|             76.2|
| 2025-11-07 11:23:22|  2025-11-10 08:40:41|       34.25|         6.54|       31.0|        4157|69.28333333333333|
| 2025-11-18 17:12:47|  2025-11-21 12:17:37|       16.55|         0.76|        9.3|        4024|67.06666666666666|
| 2025-11-22 17:45:30|  2025-11-25 09:07:36|        14.3|         1.41|       12

In [28]:
df_yellow.explain(True)

== Parsed Logical Plan ==
UnresolvedDataSource format: parquet, isStreaming: false, paths: 1 provided

== Analyzed Logical Plan ==
VendorID: int, tpep_pickup_datetime: timestamp_ntz, tpep_dropoff_datetime: timestamp_ntz, passenger_count: bigint, trip_distance: double, RatecodeID: bigint, store_and_fwd_flag: string, PULocationID: int, DOLocationID: int, payment_type: bigint, fare_amount: double, extra: double, mta_tax: double, tip_amount: double, tolls_amount: double, improvement_surcharge: double, total_amount: double, congestion_surcharge: double, Airport_fee: double, cbd_congestion_fee: double
Relation [VendorID#0,tpep_pickup_datetime#1,tpep_dropoff_datetime#2,passenger_count#3L,trip_distance#4,RatecodeID#5L,store_and_fwd_flag#6,PULocationID#7,DOLocationID#8,payment_type#9L,fare_amount#10,extra#11,mta_tax#12,tip_amount#13,tolls_amount#14,improvement_surcharge#15,total_amount#16,congestion_surcharge#17,Airport_fee#18,cbd_congestion_fee#19] parquet

== Optimized Logical Plan ==
Relatio

In [30]:
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2026-09-01 12:22:34--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 2600:9000:288b:3c00:b:20a5:b140:21, 2600:9000:288b:1000:b:20a5:b140:21, 2600:9000:288b:0:b:20a5:b140:21, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|2600:9000:288b:3c00:b:20a5:b140:21|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘taxi_zone_lookup.csv.1’

taxi_zone_lookup.cs 100%[===================>]  12.04K  --.-KB/s    in 0s      

2026-09-01 12:22:35 (124 MB/s) - ‘taxi_zone_lookup.csv.1’ saved [12331/12331]



In [32]:
df_zones = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("taxi_zone_lookup.csv")

In [33]:
df_zones.createOrReplaceTempView("zones")

In [34]:
df_zones.printSchema()

root
 |-- LocationID: integer (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- service_zone: string (nullable = true)



In [35]:
df_joined = df_yellow.join(
    df_zones, df_yellow.PULocationID == df_zones.LocationID
)

In [42]:
spark.sql("""
    SELECT
        Zone,
        COUNT(PULocationID) AS zone_frequency
    FROM joined
    GROUP BY Zone
    ORDER BY zone_frequency ASC
    LIMIT 1
    
""").show()

[Stage 17:===================>                                      (2 + 4) / 6]

+--------------------+--------------+
|                Zone|zone_frequency|
+--------------------+--------------+
|Governor's Island...|             1|
+--------------------+--------------+



In [19]:
df_yellow.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [6]:
year = 2025
month = 11

input_path = f'data/raw/yellow/{year}/{month:02d}/'
output_path = f'data/pq/yellow/{year}/{month:02d}/'

df_yellow = spark.read.parquet(input_path)

df_yellow \
    .repartition(4) \
    .write \
    .parquet(output_path)

In [4]:
df_yellow.createOrReplaceTempView("yellow")

In [7]:
spark.sql("""
SELECT COUNT(*) AS total_trips

FROM yellow
WHERE tpep_pickup_datetime >= '2025-11-15' AND tpep_pickup_datetime < '2025-11-16'

""").show()

+-----------+
|total_trips|
+-----------+
|     162604|
+-----------+



In [ ]:
spark.sql("""



""").show()

In [10]:
spark.sql("""
SELECT tpep_pickup_datetime

FROM yellow
LIMIT 10
""").show()

+--------------------+
|tpep_pickup_datetime|
+--------------------+
| 2025-11-01 00:13:25|
| 2025-11-01 00:49:07|
| 2025-11-01 00:07:19|
| 2025-11-01 00:00:00|
| 2025-11-01 00:18:50|
| 2025-11-01 00:21:11|
| 2025-11-01 00:07:31|
| 2025-11-01 00:46:52|
| 2025-11-01 00:56:59|
| 2025-11-01 00:10:43|
+--------------------+



In [4]:
df_yellow.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [5]:
df_yellow.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       7| 2025-11-01 00:13:25|  2025-11-01 00:13:25|              1|         1.68|         1|                 N|          43|    

In [4]:
from pathlib import Path

paths = [str(p) for p in Path("data/pq/green/2021").iterdir() if p.is_dir()]

df_green = spark.read.parquet(*paths)

In [5]:
from pathlib import Path

paths = [str(p) for p in Path("data/pq/yellow/2021").iterdir() if p.is_dir()]

df_yellow = spark.read.parquet(*paths)

In [10]:
spark.sparkContext.uiWebUrl

'http://10.255.255.254:4041'

In [6]:
df_green.columns

['VendorID',
 'lpep_pickup_datetime',
 'lpep_dropoff_datetime',
 'store_and_fwd_flag',
 'RatecodeID',
 'PULocationID',
 'DOLocationID',
 'passenger_count',
 'trip_distance',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'ehail_fee',
 'improvement_surcharge',
 'total_amount',
 'payment_type',
 'trip_type',
 'congestion_surcharge']

In [7]:
df_green = df_green \
    .withColumnRenamed('lpep_pickup_datetime', 'pickup_datetime') \
    .withColumnRenamed('lpep_dropoff_datetime', 'dropoff_datetime')

In [8]:
df_yellow.columns

['VendorID',
 'tpep_pickup_datetime',
 'tpep_dropoff_datetime',
 'passenger_count',
 'trip_distance',
 'RatecodeID',
 'store_and_fwd_flag',
 'PULocationID',
 'DOLocationID',
 'payment_type',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'improvement_surcharge',
 'total_amount',
 'congestion_surcharge']

In [9]:
df_yellow = df_yellow \
    .withColumnRenamed('tpep_pickup_datetime', 'pickup_datetime') \
    .withColumnRenamed('tpep_dropoff_datetime', 'dropoff_datetime')

In [24]:
common_columns = []

yellow_columns = set(df_yellow.columns)

for col in df_green.columns:
    if col in yellow_columns:
        common_columns.append(col)

In [25]:
common_columns

['VendorID',
 'pickup_datetime',
 'dropoff_datetime',
 'store_and_fwd_flag',
 'RatecodeID',
 'PULocationID',
 'DOLocationID',
 'passenger_count',
 'trip_distance',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'improvement_surcharge',
 'total_amount',
 'payment_type',
 'congestion_surcharge']

In [26]:
from pyspark.sql import functions as F

In [29]:
df_green_sel = df_green \
    .select(common_columns) \
    .withColumn('service_type', F.lit('green'))

In [30]:
df_yellow_sel = df_yellow \
    .select(common_columns) \
    .withColumn('service_type', F.lit('yellow'))

In [31]:
df_trips_data = df_green_sel.unionAll(df_yellow_sel)

In [32]:
df_trips_data.groupBy('service_type').count().show()

[Stage 6:===============================>                          (7 + 6) / 13]

+------------+--------+
|service_type|   count|
+------------+--------+
|       green|  570466|
|      yellow|15000700|
+------------+--------+



In [34]:
df_trips_data.createOrReplaceTempView('trips_data')

In [36]:
spark.sql("""
SELECT
    service_type,
    count(1)
FROM 
    trips_data
GROUP BY
    service_type

""").show()

[Stage 10:===================================================>    (12 + 1) / 13]

+------------+--------+
|service_type|count(1)|
+------------+--------+
|       green|  570466|
|      yellow|15000700|
+------------+--------+



In [37]:
df_result = spark.sql("""
SELECT 
    -- Revenue grouping 
    PULocationID AS revenue_zone,
    date_trunc('month', pickup_datetime) AS revenue_month, 
    service_type, 

    -- Revenue calculation 
    SUM(fare_amount) AS revenue_monthly_fare,
    SUM(extra) AS revenue_monthly_extra,
    SUM(mta_tax) AS revenue_monthly_mta_tax,
    SUM(tip_amount) AS revenue_monthly_tip_amount,
    SUM(tolls_amount) AS revenue_monthly_tolls_amount,
    SUM(improvement_surcharge) AS revenue_monthly_improvement_surcharge,
    SUM(total_amount) AS revenue_monthly_total_amount,
    SUM(congestion_surcharge) AS revenue_monthly_congestion_surcharge,

    -- Additional calculations
    AVG(passenger_count) AS avg_monthly_passenger_count,
    AVG(trip_distance) AS avg_monthly_trip_distance
FROM
    trips_data
GROUP BY
    revenue_zone, revenue_month, service_type
""")

In [40]:
df_result.coalesce(1).write.parquet('data/report/revenue/', mode='overwrite')

In [16]:
set(df_yellow.columns) & set(df_green.columns)

{'DOLocationID',
 'PULocationID',
 'RatecodeID',
 'VendorID',
 'congestion_surcharge',
 'extra',
 'fare_amount',
 'improvement_surcharge',
 'mta_tax',
 'passenger_count',
 'payment_type',
 'store_and_fwd_flag',
 'tip_amount',
 'tolls_amount',
 'total_amount',
 'trip_distance'}